# Client API Usage Examples

Comprehensive examples for loading full-history Kalshi project data as pandas DataFrames.

In [ ]:
# Install notebook dependencies in the active kernel environment (run once).
%pip install -q pandas requests

In [29]:
from pathlib import Path
import sys

# Make import work whether notebook is opened from repo root or from client_api/.
cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "client_api").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from client_api import KalshiDataClient
import pandas as pd

BASE_URL = "https://dehiuraiut9ro.cloudfront.net"

client = KalshiDataClient(
    base_url=BASE_URL,
    username="todd",
    password="kalshimakesmepoor",
    chunk_size=5000,
)

In [30]:
# 1) Discover API table catalog
tables = client.list_tables()
tables

['model_features_1m',
 'trade_quote_features_1m',
 'forward_quotes',
 'coinbase_ticks',
 'sim_model_predictions_1m',
 'sim_trades',
 'sim_portfolio_20s',
 'model_research_runs',
 'model_research_feature_importance']

In [31]:
# 2) Pull full history from a table (auto-pagination handled by client)
# Equivalent as below: df_features = client.fetch_table("model_features_1m"), see client.py for helper methods
df_features = client.model_features()
print(df_features.shape)
df_features.head(3)

(5857, 29)


,snapshot_minute_utc,market_ticker,event_ticker,target_btc_price,yes_bid_dollars,yes_ask_dollars,spread_dollars,mid_dollars,yes_bid_size_fp,yes_ask_size_fp,...,cb_ret_1m,cb_ret_3m,cb_ret_5m,cb_vol_5m,kalshi_data_age_sec,coinbase_data_age_sec,model_eligible,guard_status,guard_reasons_json,raw_json
0,2026-03-07 02:11:00,KXBTC15M-26MAR062115-15,KXBTC15M-26MAR062115,68271.6700,0.4100,0.4200,0.0100,0.4150,294.00,3.00,...,-0.00035034,0.00031467,0.00045778,0.00031795,0.0,0,1,ok,[],"{""trade_quote"": {""event_ticker"": ""KXBTC15M-26M..."
1,2026-03-07 02:12:00,KXBTC15M-26MAR062115-15,KXBTC15M-26MAR062115,68271.6700,0.5300,0.5400,0.0100,0.5350,608.00,325.00,...,0.00037611,0.00049484,0.00059689,0.00033922,0.0,0,1,ok,[],"{""trade_quote"": {""event_ticker"": ""KXBTC15M-26M..."
2,2026-03-07 02:13:00,KXBTC15M-26MAR062115-15,KXBTC15M-26MAR062115,68271.6700,0.6500,0.6600,0.0100,0.6550,43.00,1.00,...,0.00018396,0.00020959,0.00087498,0.00031761,0.0,0,1,ok,[],"{""trade_quote"": {""event_ticker"": ""KXBTC15M-26M..."


In [ ]:
# 3) Generic fetch with selected columns
df_preds = client.fetch_table(
    "sim_model_predictions_1m",
    columns=["snapshot_minute_utc", "model_name", "market_ticker", "prob_yes", "signal_yes", "brier", "logloss"],
)
# API JSON may return Decimal-like fields as strings; coerce to numeric for analytics.
for c in ["prob_yes", "signal_yes", "brier", "logloss"]:
    df_preds[c] = pd.to_numeric(df_preds[c], errors="coerce")

print(df_preds.shape)
df_preds.head(5)
